# Network Comparator Example

This notebook demonstrates the usage of the **network_comparator** widget, which allows displaying and comparing multiple network diagrams side-by-side with synchronized zoom and pan.


In [ ]:
import pypowsybl as pp
import pypowsybl_jupyter as ppj

## Load a network and generate diagrams

We will load a small network and create a few diagrams for comparison. For example, we can compare the same area with different layouts or different networks.


In [ ]:
n1 = pp.network.create_four_substations_node_breaker_network()
n2 = pp.network.create_four_substations_node_breaker_network() # In a real scenario, this could be a different network
n3 = pp.network.create_four_substations_node_breaker_network() # In a real scenario, this could be a different network
n4 = pp.network.create_four_substations_node_breaker_network() # In a real scenario, this could be a different network

## Display networks side-by-side

The `network_comparator` function takes a list of networks and displays their network area diagrams (NAD) on the same line. By default, zoom and pan are synchronized.


In [ ]:
ppj.network_comparator([n1, n2, n3, n4])


The toolbars in each NAD are enabled by default. Depending on the number of diagrams loaded, all those buttons can become intrusive; the parameter `display_buttons=False` removes the toolbars from the diagrams.

In [ ]:
ppj.network_comparator([n1, n2, n3, n4], display_buttons=False)

Through the optional parameters `width` and `height` it is possible to change the size of the diagrams

In [ ]:
ppj.network_comparator([n1, n2, n3, n4 ], width=400, height=400)

By default, the full diagram for the network will be displayed. However, through the optional parameters `voltage_level_ids`  and  `depth`  it is possible to display and focus only on a smaller part of the network.

In [ ]:
ppj.network_comparator([n1, n2, n3, n4 ], voltage_level_ids='S1VL1', depth=1)

## Disable synchronization

If needed, synchronization can be disabled using the `synchronized=False` parameter.


In [ ]:
ppj.network_comparator([n1, n2], synchronized=False)

## Profiles: custom labels

Profiles can be used, one for each network, for instance to set the labels for some of the branches with custom text. The profiles list can be passed to the comparator through the parameter `profiles`. A `None` entry in the list will not apply any profile to the corresponding network. Note that the text added in the examples below is for illustrative purposes only.

In [ ]:
import pandas as pd
diagram_profile1=pp.network.NadProfile(branch_labels=pd.DataFrame.from_records(index='id', columns=['id', 'middle', ], data=[('TWT', 'TWT network1 (tap=15)')]))
diagram_profile2=pp.network.NadProfile(branch_labels=pd.DataFrame.from_records(index='id', columns=['id', 'middle', ], data=[('TWT', 'TWT network2 (tap=15)')]))

ppj.network_comparator(networks=[n1, n2], profiles=[diagram_profile1, diagram_profile2])

It is also possible to change the default profile dataframes, like in this example:

In [ ]:
networks = [n1, n2, n3, n4]
profiles = [n.get_default_nad_profile() for n in networks]
for i, p in enumerate(profiles, start=1):
    p.branch_labels.at['TWT', 'middle'] = f'TWT - network{i}'
    p.branch_labels.at['HVDC1', 'middle'] = f'HVDC1 - network{i}'
    p.branch_labels.at['HVDC2', 'middle'] = f'HVDC2 - network{i}'

ppj.network_comparator(networks, profiles, display_buttons=False)

## Profiles: emphasize a list of lines

Styles in profiles could be used to emphasize a list of lines, using color and edge sizes

In [ ]:
monitored_lines = ['LINE_S2S3','LINE_S3S4']

In [ ]:
import pandas as pd
def create_nad_style_profile_for_lines(ids: list[str]) -> pd.DataFrame:
    data = [
        {
            'id': id_,
            'edge1': 'darkred',
            'width1': '12px',
            'edge2': 'darkred',
            'width2': '12px'
        }
        for id_ in ids
    ]
    return pp.network.NadProfile(edge_styles=pd.DataFrame.from_records(index='id', data=data))

In [ ]:
prof2=create_nad_style_profile_for_lines(monitored_lines)

In [ ]:
ppj.network_comparator(networks=[n1, n2], profiles=[prof2, prof2])

## Profiles: emphasize contingencies

Styles in NAD profiles could be used to emphasize contingencies: this function below, from a list of contingencies defined in a JSON string, extracts contingency elements and creates NAD profile styles to visually highlight them in NADs

In [ ]:
import json
import pandas as pd

def create_nad_style_profiles_for_contingencies(json_contingencies: str, networks: list | None = None) -> list[pp.network.NadProfile]:
    BRANCH_TYPES = {'BRANCH', 'LINE', 'HVDC_LINE', 'TWO_WINDINGS_TRANSFORMER', 'DANGLING_LINE', 'TIE_LINE'}
    INJECTION_TYPES = {'BUSBAR_SECTION', 'HVDC_CONVERTER_STATION', 'STATIC_VAR_COMPENSATOR', 'GENERATOR', 'LOAD', 'BATTERY', 'SHUNT_COMPENSATOR'}

    make_df = lambda records: None if not records else pd.DataFrame.from_records(records, index='id')

    profiles = []

    contingencies_data = json.loads(json_contingencies)

    for i, cont in enumerate(contingencies_data['contingencies']):
        grouped_elements = {'branches': [], 'buses': [], 'injections': [], 'three_wt': []}
        for el in cont['elements']:
            el_type = el.get('type')
            el_id = el['id']
            if el_type in BRANCH_TYPES:
                grouped_elements['branches'].append(el_id)
            elif el_type == 'BUS':
                grouped_elements['buses'].append(el_id)
            elif el_type == 'THREE_WINDINGS_TRANSFORMER':
                grouped_elements['three_wt'].append(el_id)
            elif el_type in INJECTION_TYPES:
                grouped_elements['injections'].append(el_id)

        # Branches
        df_branches = make_df([
            {'id': id_, 'edge1': 'yellow', 'width1': '20px', 'edge2': 'yellow', 'width2': '20px'}
            for id_ in grouped_elements['branches']
        ])

        # Buses (explicit and derived from injections)
        # connected buses for injection elements are added (when networks parametere is not None)
        bus_ids = list(grouped_elements['buses'])
        if networks is not None:
            inj_df = networks[i].get_injections(attributes=['bus_id', 'connected'])
            valid_inj_ids = [id_ for id_ in grouped_elements['injections'] if id_ in inj_df.index and inj_df.loc[id_, 'connected']]
            bus_ids = list(set(bus_ids + inj_df.loc[valid_inj_ids, 'bus_id'].tolist()))
        df_buses = make_df([
            {'id': id_, 'fill': 'yellow', 'edge': 'yellow', 'edge-width': '1px'}
            for id_ in bus_ids
        ])        

        # Three-winding transformers
        df_three_wt = make_df([
            {'id': id_, 'edge1': 'yellow', 'width1': '20px', 'edge2': 'yellow', 'width2': '20px', 'edge3': 'yellow', 'width3': '20px'}
            for id_ in grouped_elements['three_wt']
        ])

        profiles.append(pp.network.NadProfile(
            edge_styles=df_branches,
            bus_node_styles=df_buses,
            three_wt_styles=df_three_wt,
        ))

    return profiles


In [ ]:
contingencies_json = """
{
  "version" : "1.0",
  "name" : "",
  "contingencies" : [
    { "id": "id1", "elements": [ { "id": "LINE_S2S3", "type": "BRANCH" }] },
	{ "id": "id2", "elements": [ { "id": "LINE_S2S3", "type": "BRANCH" }, 
                                 { "id": "LINE_S3S4", "type": "BRANCH" }, 
                                 { "id": "HVDC1", "type": "HVDC_LINE" }, 
                                 { "id": "S3VL1_0", "type": "BUS" },
                                 { "id": "S2VL1_0", "type": "BUS" }]}
  ]
}
"""

In [ ]:
networks = [n1, n2]
ppj.network_comparator(networks=networks, profiles=create_nad_style_profiles_for_contingencies(contingencies_json, networks))